# The Hopenhayn Entry-Exit Model

# GPU

This lecture was built using a machine with JAX installed and access to a GPU.

To run this lecture on [Google Colab](https://colab.research.google.com/), click on the “play” icon top right, select Colab, and set the runtime environment to include a GPU.

To run this lecture on your own machine, you need to install [Google JAX](https://github.com/google/jax).

## Outline

I'm going to try and do this on my own, with chat GPT, so I can understand every step of the way. Here is some code that we have:

We will use the following imports.

In [6]:
import numpy as np
import scipy.optimize as opt

from scipy.stats import norm

In [7]:
# Parameters

beta = 0.95           # Discount factor
sigma_eps = 0.2       # Shock standard deviation
rho = 0.9             # Persistence of productivity
N = 100               #     Number of productivity grid points
z_min, z_max = -2, 2  # Bounds for log-productivity

Here is some code that allows us to discretize an AR(1) process:

In [8]:
def tauchen(N, mu, rho, sigma, n_std=3):
    z = np.linspace(mu - n_std * sigma / np.sqrt(1 - rho**2),
                     mu + n_std * sigma / np.sqrt(1 - rho**2), N)
    step = (z[1] - z[0])
    P = np.zeros((N, N))

    for j in range(N):
        for k in range(N):
            if k == 0:
                P[j, k] = norm.cdf((z[k] - rho * z[j] + step / 2) / sigma)
            elif k == N-1:
                P[j, k] = 1 - norm.cdf((z[k] - rho * z[j] - step / 2) / sigma)
            else:
                P[j, k] = (norm.cdf((z[k] - rho * z[j] + step / 2) / sigma) -
                           norm.cdf((z[k] - rho * z[j] - step / 2) / sigma))

    return np.exp(z), P

Just to see how this works, let's plug in some of the values that we initialized:

In [9]:
z_grid, P_z = tauchen(N, 0, rho, sigma_eps)

In [14]:
# Now, let's print some info about the grid (you can modify this)
print("Shape of the productivity grid:", z_grid.shape)
print("First 5 values of z_grid:", z_grid[:5])

# And the transition matrix
print("\nShape of the transition matrix P_z:", P_z.shape)


Shape of the productivity grid: (100,)
First 5 values of z_grid: [0.25246203 0.25958101 0.26690074 0.27442686 0.28216521]

Shape of the transition matrix P_z: (100, 100)


So, we now have our transition matrix for the productivity grid. Nice. Now, what we need to do is define a profit function, which we just assume is given by productivity minus 1. This is pretty easy to deal with!

In [15]:
def profits(z):
  return z - 1

Next, we want to solve the value function...which I'm not sure is done correctly. Anyways, we have:

In [16]:
def solve_value_function():
    V = np.zeros(N)
    tol = 1e-6
    diff = 1

    while diff > tol:
        V_new = np.maximum(0, profits(z_grid) + beta * P_z @ V)  # Bellman operator
        diff = np.max(np.abs(V_new - V))
        V = V_new

    return V

Note that the above should return a vector; also note that the @ operator is for matri multiplication. Therefore, we are multiplying the transition matrix out.

One thing I don't love about this code is that it pulls in ```N``` and ```z_grid``` from elsewhere. That might be a way to clean up things a little bit.

In [17]:
V = solve_value_function()

In [18]:
V

array([ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.06109435,  0.14631188,  0.23810126,
        0.33645764,  0.44132982,  0.5526275 ,  0.67022931,  0.79399124,
        0.92375484,  1.05935481,  1.2006257 ,  1.34740746,  1.49954981,
        1.65691534,  1.81938159,  1.98684205,  2.15920649,  2.33640056,
        2.51836508,  2.70505497,  2.8964381 ,  3.09249406,  3.293213  ,
        3.49859451,  3.70864662,  3.92338492,  4.1428317 ,  4.36701532,
        4.59596949,  4.82973275,  5.06834791,  5.31186152,  5.56032338,
        5.81378604,  6.07230424,  6.33593439,  6.60473396,  6.87

Here is the exit threshold:

In [19]:
z_exit_idx = np.where(V == 0)[0][0]

In [21]:
z_exit = z_grid[z_exit_idx]

Evidently, whenever productivity falls below ```z_exit``` one should leave the industry.

In [22]:
z_exit

0.25246203368307146

Stationary distribution is as follows: